# **Atividade Prática**
<font size=3>

- **Tema:** Análise semântica.
- **Prazo de entrega:** 29 de Março.

**Envie** a atividade pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=publish-editor).

---

## **Enunciado:**
<font size=3>

Utilizando o [conjunto de dados de notícias de jornal](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html), desenvolva os ítens abaixo.


### **1. Importação de dados:**
<font size=3>

Defina o objeto do conjunto de dados com a classe `fetch_20newsgroups`,
 - Ao definir a classe `fetch_20newsgroups`, **remova** os cabeçalhos (*headers*), rodapés (*footers*) e citações (*quotes*), com a variável `remove=('headers', 'footers', 'quotes')`.
 - **Imprima** na tela o primeiro texto de notícia.

In [13]:
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

# importando o dataset de notícias com duas categorias:
dataset = fetch_20newsgroups(remove=('headers', 'footers', 'quotes'))
texts = dataset.data

textsDF = pd.DataFrame(texts)

print("Primeiros 10 textos:")
print(textsDF.head(10))


Primeiros 10 textos:
                                                   0
0  I was wondering if anyone out there could enli...
1  A fair number of brave souls who upgraded thei...
2  well folks, my mac plus finally gave up the gh...
3  \nDo you have Weitek's address/phone number?  ...
4  From article <C5owCB.n3p@world.std.com>, by to...
5  \n\n\n\n\nOf course.  The term must be rigidly...
6  There were a few people who responded to my re...
7                                                ...
8  I have win 3.0 and downloaded several icons an...
9  \n\n\nI've had the board for over a year, and ...


In [14]:
print("Número de textos:")
print(textsDF.shape)

Número de textos:
(11314, 1)


### **2. Pré-propressamento:**
<font size=3>

Define uma função que **receba um texto** e o **retorne pré-processado**, dado pelas seguintes etapas:
- Converter os caracteres em minúsculo;
- Remover as pontuações com a expressão regular `[^\w\s]` e função [sub( )](https://docs.python.org/3/library/re.html#re.sub);
- Remover dígitos sequenciais com a expressão regular `\d+`;
- Remover *underline* em sequência com a expressão regular `_+`;
- Remover espaços em branco em sequência com a expressão regular `\s+`;
- Remoção de [*stopword*](https://pythonspot.com/nltk-stop-words/);
- [Lematização](https://www.nltk.org/api/nltk.stem.WordNetLemmatizer.html?highlight=wordnet).

**Para testar** sua função **imprima na tela** o primeiro texto *antes* e *depois* do pré-processamento.

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import re

lemmatizer = WordNetLemmatizer()

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocessor(text):
  out = text.lower()
  out = re.sub(r'[^\w\s]', '', out)
  out = re.sub(r'\d+', '', out)
  out = re.sub(r'_+', '', out)
  out = re.sub(r'\s+', ' ', out)
  out = ' '.join([token for token in out.split() if token not in stop_words])
  out = lemmatizer.lemmatize(out)
  return out

print("Texto original:")
print(texts[0])
print()
print("Texto pré-processado:")
print(preprocessor(texts[0]))


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Texto original:
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Texto pré-processado:
wondering anyone could enlighten car saw day door sports car looked late early called bricklin doors really small addition front bumper separate rest body know anyone tellme model name engine specs years production car made history whatever info funky looking car please email


### **3. Vetorização:**
<font size=3>

Faça a [vetorização TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) do textos pré-processados. Declare sua função de pré-processamento ao definir a classe `TfidfVectorizer`, e utilize os argumentos `max_df` e `min_df` para filtrar o vocabulário.

- **Imprima** as 10 primeiras palavras do vocabulário, usando o método `get_feature_names_out()`;
- **Imprima** o tamanho do vocabulário.
  

In [16]:
vectorizer = TfidfVectorizer(preprocessor=preprocessor, max_df=0.8, min_df=100)

tfidf_matrix = vectorizer.fit_transform(texts)

terms = vectorizer.get_feature_names_out()

doc_names = [f'Doc{i+1}' for i in range(len(texts))]

df = pd.DataFrame(tfidf_matrix.T.toarray(), index=terms, columns=doc_names)

In [17]:
print(f"Tamanho do vocabulário: {df.shape[0]}")
n = 20
print(f"Primeiras {n} palavras do vocabulário:")
for term in df.index[:n]:
  print(term)


Tamanho do vocabulário: 1365
Primeiras 20 palavras do vocabulário:
ability
able
absolutely
accept
accepted
access
according
account
accurate
across
act
action
actions
acts
actual
actually
ad
add
added
addition


#### **3.1 Palavras-chaves:**
<font size=3>

Escolha um documento (um texto) e imprima na tela suas **palavras-chaves** (os termos mais frequentes do codumento).

Para isso:
 - Com o objeto da transformação TF-IDF, acesse o _array_ com o método `toarray()`;
 - Selecione uma das linhas (um documento) desse _array_;
 - Obtenha os índices ordenados desse vetor com a função [np.argsort()](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html);
 - Aplique os índices no _array_ de **vocabulário** para se obter os termos ordenados de acordo com o vetor;
 - **Imprima na tela** as 5 palavras mais frequentes desse documento.


In [18]:
import numpy as np

print("Texto original:")
print(texts[0])
print()
print("Texto pré-processado:")
print(preprocessor(texts[0]))

doc1_tfidf = tfidf_matrix.toarray()[0]
doc1_sorted_indices = np.argsort(doc1_tfidf)[::-1]

print()
print (f"Shape tfidf_matrix: {tfidf_matrix.toarray().shape}")
print (f"Doc1 tfidf: {doc1_tfidf}")
print (f"Doc1 sorted indices: {doc1_sorted_indices}")

print()
for rank, i in enumerate(doc1_sorted_indices, start=1):
    if (rank > 5):
        break
    if doc1_tfidf[i] > 0:
        print(f"Top N frequência: {rank} \tPosição vocabulário: {i} \tTermo: {terms[i]}\tTFIDF: {doc1_tfidf[i]}")


Texto original:
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Texto pré-processado:
wondering anyone could enlighten car saw day door sports car looked late early called bricklin doors really small addition front bumper separate rest body know anyone tellme model name engine specs years production car made history whatever info funky looking car please email

Shape tfidf_matrix: (11314, 1365)
Doc1 tfidf: [0. 0. 0. ... 0. 0. 0.]
Doc1 sorted indices: [172  58 339 ... 909 910   0]

Top N frequência: 1 	Posição vocabulário: 172 	Termo: car	TFIDF: 0.5838973601992797
T

### **4. Análise semântica:**
<font size=3>

A Análise Semântica Latente é desenvolvida em três etapas:
 1. Criar a matriz termo-documento $A$;
 2. Fazer a Decomposição em Valores Singulares e truncar em $k$ componentes, dado por:
   $$
        A \approx A_k = U_k\cdot \Sigma_k\cdot V_k^T \,
   $$
 3. Criar o espaço semântico latente a partir das projeções das matrizes fatoradas.

**Imprima na tela** a matriz **termo-documento** como um [*dataframe*](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html), com base na vetorização TF-IDF.

In [ ]:
tfidf_array = tfidf_matrix.toarray()
print(f"tfidf_array.shape: {tfidf_array.shape}")

tfidf_matrix_np = np.array(tfidf_array) # converte a matriz tfidf para um array numpy

print(f"tfidf_matrix_np.shape: {tfidf_matrix_np.shape}")
A = tfidf_matrix_np.T # transposta da matriz tfidf_np

print(f"termo_documento.shape: {A.shape}")

termo_documento_df = pd.DataFrame(A, index=terms, columns=doc_names)
termo_documento_df


tfidf_array.shape: (11314, 1365)
tfidf_matrix_np.shape: (11314, 1365)
termo_documento.shape: (1365, 11314)


,Doc1,Doc2,Doc3,Doc4,Doc5,Doc6,Doc7,Doc8,Doc9,Doc10,...,Doc11305,Doc11306,Doc11307,Doc11308,Doc11309,Doc11310,Doc11311,Doc11312,Doc11313,Doc11314
ability,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
able,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.051907,0.0,0.0,0.0,0.095286,0.0,0.0,0.0,0.0
absolutely,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.130516,0.0,0.0,0.0,0.0
accept,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.064815,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
accepted,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
youd,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
youll,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
young,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.136741,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
youre,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


#### **4.1 Espaço semântico de termos:**
<font size=3>

Faça a **Decomposição em Valores Singulares** truncando em **2 componentes**, usando a classe [TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html), a fim de obter a projeção **termo-conceito**, $W_k = U_k\cdot \Sigma_k$.

- **Plote** 5 termos da projeção $W_k$ no espaço semântico bidimensional.

> **Dica:** você pode usar uma combinação de plote [scatter](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) e [text](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.text.html) em um *loop*.

#### **4.2 Similaridade entre termos:**
<font size=3>

- Calcule a [similaridade de cosseno](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) entre os pares de palavras ("love", "fly"), ("love", "run") e ("fly", "run").
  > **Dica:** Busque tais palavras no *array* de **vocabulário** usando a [`np.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html). Em seguinda, aplique a métrica à projeção $W_k$.

- **Escreva** em uma célula **markdown** sua interpretação dos valores dos resultados.
